# Прогнозирование CTR и калибровка вероятностей в рекламных аукционах Advandex

## Описание проекта

В рамках проекта разрабатывается модель бинарной классификации для предсказания вероятности клика (CTR) по рекламному объявлению в системе реального времени.

Модель используется в механизме рекламных аукционов. Победитель аукциона определяется на основе ожидаемой ценности показа, рассчитываемой по формуле:

Ставка × Предсказанный CTR

Таким образом, корректность предсказанных вероятностей напрямую влияет на финансовые результаты платформы и рекламодателей.

Завышенные вероятности приводят к переплатам за показы.
Заниженные вероятности приводят к потере дохода и проигрышу аукционов.

Поэтому модель должна быть не только точной, но и откалиброванной — предсказанные вероятности должны соответствовать реальной частоте кликов.

---

## Цель проекта

1. Построить модель бинарной классификации для предсказания вероятности клика (CTR).
2. Оценить качество модели с использованием метрик:
   - PR-AUC (основная метрика для несбалансированных данных),
   - Log Loss,
   - Brier Score.
3. Провести калибровку вероятностей модели (изотоническая регрессия).
4. Оценить улучшение калибровки с помощью Brier Score и диаграмм калибровки.
5. Подготовить воспроизводимое и готовое к продакшену решение с использованием пайплайнов.

# Структура проекта

## 1. Подготовка среды и загрузка данных

#### 1.1 Подготовьте библиотеки
- Создайте файл `requirements.txt` с фиксированными версиями всех пакетов.
- Импортируйте все необходимые библиотеки.
- Настройте параметры отображения графиков и датафреймов.

#### 1.2 Зафиксируйте константу для воспроизводимости
- Установите константу `RANDOM_SEED`.
- Применяйте её ко всем алгоритмам, которые её поддерживают.

#### 1.3 Загрузите данные
- Прочитайте CSV-файл с данными. Путь к файлу: `'/datasets/ds_s16_ad_click_dataset.csv'`
- Выведите размер датасета, первые несколько строк и информацию о типах столбцов.
- Проверьте успешность загрузки данных.

In [ ]:
!pip install category-encoders==2.2.2
!pip install phik==0.12.5

In [ ]:
import pandas, numpy, sklearn, matplotlib, seaborn, scipy, joblib, phik
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("sklearn:", sklearn.__version__)
print("matplotlib:", matplotlib.__version__)
print("seaborn:", seaborn.__version__)
print("scipy:", scipy.__version__)
print("joblib:", joblib.__version__)
print("phik:", phik.__version__)

In [ ]:
requirements = """pandas==1.2.4
numpy==1.21.1
scikit-learn==0.24.1
matplotlib==3.3.4
seaborn==0.11.1
scipy==1.9.1
joblib==1.1.0
category-encoders==2.2.2
phik==0.12.5
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

In [ ]:
import numpy as np
import pandas as pd
import json

import matplotlib.pyplot as plt
import seaborn as sns

from phik import phik_from_array
from phik.report import plot_correlation_matrix

from sklearn.svm import LinearSVC
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_validate
)
from sklearn.feature_selection import VarianceThreshold, RFECV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from category_encoders import TargetEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
    brier_score_loss
)
from sklearn.isotonic import IsotonicRegression
from scipy.special import expit
from sklearn.calibration import (
    CalibratedClassifierCV,
    calibration_curve
)
from sklearn.exceptions import ConvergenceWarning

import joblib

import warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore")

In [ ]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

plt.style.use("seaborn")
sns.set_context("notebook")

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

%matplotlib inline

In [ ]:
RANDOM_SEED = 42
RANDOM_STATE = RANDOM_SEED
np.random.seed(RANDOM_SEED)

In [ ]:
file_path = "/datasets/ds_s16_ad_click_dataset.csv"

df = pd.read_csv(file_path)

In [ ]:
print("Размер датасета:", df.shape)

In [ ]:
display(df.head())

In [ ]:
print("\nИнформация о типах данных:")
df.info()

- Подготовлена среда: создан `requirements.txt`, импортированы библиотеки, настроено отображение таблиц и графиков.
- Зафиксирована константа `RANDOM_SEED` для воспроизводимости и будет использоваться во всех методах, поддерживающих `random_state`.
- Данные успешно загружены из `/datasets/ds_s16_ad_click_dataset.csv`.
- Размер датасета: 50000 строк и 34 столбца.
- Пропуски отсутствуют

In [ ]:
df = df.copy()

dt = pd.to_datetime(df["hour"].astype(str), format="%y%m%d%H", errors="coerce")

df["hour_of_day"] = dt.dt.hour.astype("Int64")
df["dayofweek"] = dt.dt.dayofweek.astype("Int64")
df["is_weekend"] = (df["dayofweek"] >= 5).astype("Int64")

df = df.drop(columns=["hour"])

display(df[["hour_of_day", "dayofweek", "is_weekend"]].head())
print(df.columns.tolist())
print(df[["hour_of_day", "dayofweek", "is_weekend"]].isna().mean())

<div class="alert alert-info">

<h2>Комментарий студента<a class="tocSkip"></a></h2>

Признак <code>hour</code> в исходных данных представлен в формате YYMMDDHH и содержит абсолютную временную метку.  
Использование такого представления может приводить к искажению временной зависимости при обучении модели.

Признак был преобразован в интерпретируемые компоненты:
<ul>
<li><code>hour_of_day</code> - час суток</li>
<li><code>dayofweek</code> - день недели</li>
<li><code>is_weekend</code> - индикатор выходного дня</li>
</ul>

Исходный признак <code>hour</code> удален. Проверка показала, что созданные признаки успешно добавлены в датасет и не содержат пропусков.  
Дальнейший анализ и моделирование выполняются на обновленном наборе признаков.

</div>

## 2. Исследовательский анализ данных (EDA)

#### 2.1 Опишите базовую информацию о датасете
- Определите, сколько объектов и признаков содержится в данных.
- Выясните, какие типы данных представлены (числовые, категориальные).
- Дайте общее описание: укажите, что известно о пользователях и рекламе.

#### 2.2 Анализ целевой переменной
- Проанализируйте, как распределена целевая переменная.
- Определите, есть ли дисбаланс классов. Это важно для выбора метрик.
- Посчитайте долю рекламы, на которую кликнули, и долю рекламы, на которую не кликнули.

#### 2.3 Анализ признаков
- Определите, все ли признаки нужны для обучения модели. Есть ли среди них явно бесполезные?
- Опишите, какие признаки категориальные, а какие — числовые.
- Проведите первичный отбор: удалите ненужные признаки.

#### 2.4 Анализ пропущенных значений
- Проверьте долю пропусков в каждом признаке.
- Выберите корректную стратегию для заполнения пропусков — удаление, среднее, медиана, мода. Выбор обоснуйте.

#### 2.5 Анализ категориальных признаков
- Определите, сколько уникальных значений в каждом категориальном признаке.
- Укажите, какие признаки можно кодировать One-Hot Encoding, а какие требуют специальных методов из-за высокой кардинальности.

#### 2.6 Анализ выбросов и распределений
- Проверьте, есть ли явные выбросы в числовых признаках.
- Опишите, как распределены выбросы — нормально, асимметрично, каким-то другим образом.

#### 2.7 Корреляции
- Определите, какие признаки коррелируют с целевой переменной.
- Выявите сильно скоррелированные признаки, которые можно удалить, если такие есть.

#### 2.8 Выводы по EDA
- Кратко опишите ключевые находки.
- Выберите признаки, которые выглядят наиболее перспективными для модели. Выбор обоснуйте.
- Определите действия по предобработке данных, которые необходимо проделать.

In [ ]:
target = "click"

print("Объектов (строк):", df.shape[0])
print("Признаков (столбцов):", df.shape[1])

features = [c for c in df.columns if c != target]
cat_cols = df[features].select_dtypes(include=["object"]).columns.tolist()
num_cols = df[features].select_dtypes(include=[np.number]).columns.tolist()

print("Категориальные признаки (object):", len(cat_cols))
print("Числовые признаки (number):", len(num_cols))

display(pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values
}))

- Датасет содержит 50000 объектов и 36 признака.
- Целевая переменная - `click` (бинарный признак).
- По типам данных:
  - 11 признаков имеют тип `object` (категориальные в явном виде),
  - 24 признака имеют числовой тип (`int64` и `float64`).

Однако следует учитывать семантику признаков:
- `id` является идентификатором и не будет использоваться в обучении модели.
- `hour` является временной меткой и требует преобразования.
- Часть числовых признаков (`device_type`, `device_conn_type`, `C1`, `C14–C21`) по смыслу являются категориальными и будут обрабатываться соответствующим образом.

Данные содержат информацию о:
- рекламной площадке (site_*),
- рекламируемом приложении (app_*),
- характеристиках устройства пользователя (device_*),
- параметрах баннера и аукциона (C*),
- машинно-сгенерированных признаках (ml_feature_*).

In [ ]:
target = "click"

ctr = df[target].mean()
counts = df[target].value_counts()
shares = df[target].value_counts(normalize=True)

print(f"CTR (доля кликов) = {ctr:.6f}")

display(
    pd.DataFrame({"count": counts, "share": shares})
    .sort_index()
)

ax = shares.sort_index().plot(kind="bar")
ax.set_title("Распределение целевой переменной click")
ax.set_xlabel("click")
ax.set_ylabel("Доля")
plt.show()

- Доля кликов (CTR) составляет 0.1721 (17.21%).
- Доля показов без клика - 82.79%.
- В абсолютных значениях:
  - 41397 наблюдений без клика,
  - 8603 наблюдения с кликом.

Наблюдается выраженный дисбаланс классов: класс "0" существенно преобладает над классом "1".

Это подтверждает необходимость использования метрик, устойчивых к дисбалансу классов. В качестве основной метрики будет использоваться PR-AUC, поскольку она лучше отражает способность модели находить редкий положительный класс (клики).

Дополнительно будут использоваться Log Loss и Brier Score для оценки качества вероятностных прогнозов и калибровки модели.

In [ ]:
drop_cols = ["id"]
df_eda = df.drop(columns=drop_cols).copy()

print("Размер после удаления id:", df_eda.shape)

semantic_cat = [
    "hour_of_day",
    "dayofweek",
    "is_weekend",
    "C1",
    "banner_pos",
    "device_type",
    "device_conn_type",
    "C14","C15","C16","C17","C18","C19","C20","C21"
]

semantic_cat = [c for c in semantic_cat if c in df_eda.columns]

target = "click"
features = [c for c in df_eda.columns if c != target]

cat_cols_obj = df_eda[features].select_dtypes(include=["object"]).columns.tolist()
num_cols_raw = df_eda[features].select_dtypes(include=[np.number]).columns.tolist()

num_cols = [c for c in num_cols_raw if c not in semantic_cat]
cat_cols = sorted(set(cat_cols_obj + semantic_cat))

print("Числовые признаки (для scaler):", len(num_cols))
print("Категориальные признаки (для encoder):", len(cat_cols))

print("\nПример числовых:", num_cols[:5])
print("Пример категориальных:", cat_cols[:5])

- Признак `id` удалён, так как является идентификатором и не несёт предсказательной информации.
- После удаления `id` в датасете осталось 35 признака.

Признаки были разделены с учётом их семантики:

- 8 числовых признаков (в основном машинно-сгенерированные `ml_feature_*`), которые будут масштабироваться.
- 24 категориальных признака, включая:
  - признаки площадки (`site_*`),
  - признаки приложения (`app_*`),
  - признаки устройства (`device_*`),
  - параметры баннера и аукциона (`C*`, `banner_pos`),
  - временной признак `hour`.

Отдельно отмечено, что часть признаков, представленных как `int64` (например, `C1`, `C14–C21`, `device_type`, `device_conn_type`), по смыслу являются категориальными и будут обрабатываться соответствующим образом.

Таким образом, сформированы списки признаков для дальнейшей предобработки:
- числовые - для масштабирования,
- категориальные - для кодирования.

In [ ]:
na_rate = df_eda.isna().mean().sort_values(ascending=False)

display(na_rate[na_rate > 0])

print("Общее количество пропусков:", df_eda.isna().sum().sum())

Пропущенные значения в датасете отсутствуют. Все признаки содержат 50000 непустых значений.

Несмотря на отсутствие пропусков в текущем наборе данных, в пайплайне предобработки будет предусмотрена обработка пропущенных значений:

- для числовых признаков - заполнение медианой,
- для категориальных признаков - заполнение отдельной категорией (например, "missing").

Это необходимо для обеспечения устойчивости модели в продакшене, где новые данные могут содержать пропуски.

In [ ]:
cardinality = df_eda[cat_cols].nunique(dropna=False).sort_values(ascending=False)

display(cardinality)

print("\nМаксимальная кардинальность:", cardinality.max())
print("Минимальная кардинальность:", cardinality.min())

Категориальные признаки существенно различаются по кардинальности.

**Признаки с очень высокой кардинальностью (более 1000 уникальных значений):**  
`device_ip`, `device_id`, `device_model`, `site_id`, `site_domain`, `app_id`, `C14`.  
Для этих признаков применение One-Hot Encoding приведет к резкому росту размерности разреженной матрицы. Для них будет использоваться Target Encoding.

**Признаки со средней кардинальностью (20-400 уникальных значений):**  
`C17`, `C20`, `app_domain`, `C19`, `C21`, `app_category`, `site_category`, `hour_of_day`.  
Для них возможны разные подходы, однако для снижения размерности и поддержания компактности модели также будет использоваться Target Encoding.

**Признаки с низкой кардинальностью (до 20 уникальных значений):**  
`C16`, `C15`, `C1`, `banner_pos`, `ml_feature_2`, `device_type`, `device_conn_type`, `C18`, `ml_feature_7`, `dayofweek`, `is_weekend`.  
Эти признаки будут кодироваться с помощью One-Hot Encoding.

<div class="alert alert-info">

<h2>Комментарий студента<a class="tocSkip"></a></h2>

<b>Исправлено:</b><br/>
Преобразование признака <code>hour</code> выполнено до этапа EDA.  
Исходный признак в формате YYMMDDHH преобразован в:
<ul>
<li><code>hour_of_day</code> - час суток</li>
<li><code>dayofweek</code> - день недели</li>
<li><code>is_weekend</code> - индикатор выходного дня</li>
</ul>

Исходный признак <code>hour</code> удален.  
Дальнейший анализ выполняется на обновленном наборе признаков.

</div>

In [ ]:
display(df_eda[num_cols].describe().T)

In [ ]:
df_eda[num_cols].hist(bins=30, figsize=(14, 10))
plt.suptitle("Распределения числовых признаков", y=1.02)
plt.show()

In [ ]:
outlier_share = {}

for col in num_cols:
    q1 = df_eda[col].quantile(0.25)
    q3 = df_eda[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    
    share = ((df_eda[col] < lower) | (df_eda[col] > upper)).mean()
    outlier_share[col] = share

outlier_share = pd.Series(outlier_share).sort_values(ascending=False)

display(outlier_share)

В датасете выделено 8 числовых признаков (в основном `ml_feature_*`).

Распределения:

- Большинство признаков (`ml_feature_1`, `ml_feature_5`, `ml_feature_9`, `ml_feature_10`) имеют близкое к нормальному распределение.
- `ml_feature_3` распределён равномерно в диапазоне примерно [-10, 10].
- `ml_feature_4` является бинарным признаком (0/1).
- `ml_feature_6` и `ml_feature_8` ограничены диапазоном [-1, 1] и имеют симметричную форму.

Выбросы:

- Доля выбросов по IQR-критерию не превышает 0.8% для любого признака.
- Существенных экстремальных значений не обнаружено.

Вывод:

- Числовые признаки уже масштабированы и находятся в сопоставимых диапазонах.
- Дополнительные преобразования (логарифмирование, клиппинг) не требуются.
- Для модели SVM будет достаточно применения StandardScaler в пайплайне.

In [ ]:
uniq_ratio = (df_eda.nunique(dropna=False) / len(df_eda)).sort_values(ascending=False)
display(uniq_ratio.head(20))

In [ ]:
target = df_eda["click"].values

uniq_ratio = df_eda.nunique(dropna=False) / len(df_eda)

bad_cols = uniq_ratio[uniq_ratio >= 0.95].index.tolist()

cols = [c for c in df_eda.columns if c not in (["click", "id"] + bad_cols)]

scores = {c: phik_from_array(df_eda[c].values, target) for c in cols}
phik_with_target = pd.Series(scores).sort_values(ascending=False)

print("Исключены признаки:", bad_cols)
display(phik_with_target.head(20))

<div class="alert alert-info">

<h2>Комментарий студента<a class="tocSkip"></a></h2>

<b>Исправлено:</b><br/>
Корреляция Пирсона заменена на коэффициент φ_k, поскольку целевая переменная <code>click</code> является бинарной, а распределения признаков не являются нормальными.

Расчет выполнен только для пар «признак - click» без построения полной φ_k-матрицы, что позволяет избежать избыточных вычислений при высокой кардинальности признаков.

Дополнительно из анализа исключены признаки с долей уникальных значений, близкой к 1 (часть <code>ml_feature_*</code>), так как они фактически являются идентификатороподобными и приводят к артефактным значениям φ_k.

</div>

In [ ]:
uniq_ratio = df_eda.nunique(dropna=False) / len(df_eda)
bad_cols = uniq_ratio[uniq_ratio >= 0.95].index.tolist()

cols = [c for c in df_eda.columns if c not in (["click", "id"] + bad_cols)]

target = df_eda["click"].values
scores = {c: phik_from_array(df_eda[c].values, target) for c in cols}
phik_with_target = pd.Series(scores).sort_values(ascending=False)

high_card_cols = df_eda.nunique()[df_eda.nunique() > 1000].index.tolist()

top_features = [
    c for c in phik_with_target.head(20).index.tolist()
    if c not in high_card_cols
][:12]

cols_for_matrix = top_features + ["click"]

interval_cols = [
    c for c in cols_for_matrix
    if c.startswith("ml_feature_") and pd.api.types.is_numeric_dtype(df_eda[c])
]

phik_overview = df_eda[cols_for_matrix].phik_matrix(interval_cols=interval_cols)

plot_correlation_matrix(
    phik_overview.values,
    x_labels=phik_overview.columns,
    y_labels=phik_overview.index,
    title=r"Top features correlation $\phi_K$",
    fontsize_factor=1.2,
    figsize=(10, 8)
)

<div class="alert alert-info">

<h2>Комментарий студента<a class="tocSkip"></a></h2>

<b>Исправлено:</b><br/>
Корреляционная матрица на основе коэффициента Пирсона заменена на матрицу коэффициентов φ_k.

Визуализация выполнена для наиболее информативных признаков (top по φ_k относительно целевой переменной <code>click</code>).  
Использование φ_k позволяет учитывать нелинейные зависимости и корректно работать с признаками различных типов без предварительного кодирования.

</div>

Корреляции между числовыми признаками:

- Между числовыми признаками отсутствуют сильные корреляции.
- Максимальные значения находятся в пределах |0.02|, что свидетельствует об отсутствии мультиколлинеарности.
- Удаление числовых признаков по причине сильной взаимосвязи не требуется.

Корреляция числовых признаков с целевой переменной:

Наибольшую линейную связь с целевой переменной `click` демонстрируют:

- `ml_feature_9` (≈ 0.146)
- `ml_feature_10` (≈ 0.131)
- `ml_feature_8` (≈ 0.080)
- `ml_feature_6` (≈ 0.072)
- `ml_feature_5` (≈ 0.063)

Остальные числовые признаки имеют близкую к нулю линейную корреляцию.

Вывод:

- Числовые признаки не дублируют друг друга.
- Часть `ml_feature_*` демонстрирует умеренную линейную связь с вероятностью клика.
- Удаление числовых признаков на основании корреляционного анализа не требуется.

### Выводы по EDA

В ходе исследовательского анализа данных получены следующие результаты:

### Структура данных
- Датасет содержит 50000 объектов и 34 признака.
- Целевая переменная `click` является бинарной.
- После удаления идентификатора `id` для моделирования остаётся 33 признака.

### Дисбаланс классов
- Доля кликов (CTR) составляет 17.21%.
- Наблюдается выраженный дисбаланс классов.
- В качестве основной метрики будет использоваться PR-AUC.
- Дополнительно будут использоваться Log Loss и Brier Score для оценки вероятностей и калибровки.

### Признаки
- Выделено 8 числовых признаков (`ml_feature_*`).
- 24 признака являются категориальными (включая семантически категориальные `C*`, `device_*`, `hour`).

### Пропуски
- Пропущенные значения отсутствуют.
- В пайплайне предобработки будет предусмотрена обработка пропусков для устойчивости модели.

### Категориальные признаки
- Признаки с низкой кардинальностью будут кодироваться с помощью One-Hot Encoding.
- Признаки с высокой кардинальностью (`device_ip`, `device_id`, `site_id`, `site_domain`, `app_id`, `C14` и др.) требуют Target Encoding.
- One-Hot Encoding для них приведёт к чрезмерному росту размерности.

### Числовые признаки
- Распределения в целом симметричны.
- Существенных выбросов не обнаружено.
- Дополнительные нелинейные преобразования не требуются.
- Для SVM будет применён StandardScaler.

### Корреляционный анализ
- Сильной корреляции между числовыми признаками не обнаружено.
- Наиболее информативные признаки среди числовых: `ml_feature_9`, `ml_feature_10`, `ml_feature_8`.
- Удаление признаков по причине мультиколлинеарности не требуется.

---

## План дальнейшей предобработки

1. Обработка временного признака `hour`.
2. Разделение категориальных признаков по кардинальности.
3. Применение:
   - One-Hot Encoding для low-cardinality,
   - Target Encoding для high-cardinality.
4. Масштабирование числовых признаков.
5. Объединение всех шагов в единый Pipeline.
6. Обучение моделей:
   - Logistic Regression (baseline),
   - SVM (основная модель).
7. Калибровка вероятностей методом изотонической регрессии.

## 3. Разделение данных на выборки

#### 3.1 Разделите данные
- Сначала отделите тестовую выборку, в ней должно быть 20% данных.
- Оставшиеся 80% данных используйте для обучения.
- Используйте стратифицированное разделение, чтобы сохранить баланс классов.
- **Не используйте тестовую выборку до финального тестирования!**

#### 3.2 Проверьте разделение
- Убедитесь, что распределение целевой переменной сохранено в каждой выборке.
- Выведите размеры выборок.

In [ ]:
target = "click"

X = df_eda.drop(columns=[target])
y = df_eda[target]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.4,
    stratify=y,
    random_state=RANDOM_SEED
)

X_cal, X_test, y_cal, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=RANDOM_SEED
)

print("Размер X_train:", X_train.shape)
print("Размер X_cal:", X_cal.shape)
print("Размер X_test:", X_test.shape)

In [ ]:
print("Распределение в полном датасете:")
display(y.value_counts(normalize=True))

print("\nРаспределение в обучающей выборке:")
display(y_train.value_counts(normalize=True))

print("\nРаспределение в тестовой выборке:")
display(y_test.value_counts(normalize=True))

После стратифицированного разделения распределение целевой переменной в обучающей и тестовой выборках полностью сохранено.

Доля кликов:

- В полном датасете: 17.206%
- В обучающей выборке: 17.205%
- В тестовой выборке: 17.210%

Отклонения находятся в пределах округления, что подтверждает корректность стратифицированного разделения.

Тестовая выборка не будет использоваться до финальной оценки модели.

## 4. Предобработка данных — построение пайплайнов

#### 4.1 Создайте пайплайн для предобработки данных

**Для числовых признаков:**
- Корректно заполните пропуски — средним, медианой или другим методом.
- Масштабируйте данные с помощью `StandardScaler`.
- Обработайте выбросы, если необходимо.

**Для категориальных признаков:**
- Корректно заполните пропуски — значением по умолчанию или модой.
- Примените кодирование:
  - One-Hot Encoding для признаков с малой кардинальностью.
  - Target Encoding для признаков с высокой кардинальностью.

#### 4.2 Объедините пайплайны
- Используйте `sklearn.pipeline.Pipeline` и `ColumnTransformer`.
- **Важно:** используйте информацию о пропусках и категориях только из обучающей выборки!

In [ ]:
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()

card = X_train[cat_cols].nunique().sort_values(ascending=False)

thr = 20
cat_high_cols = card[card > thr].index.tolist()
cat_low_cols = card[card <= thr].index.tolist()

print("Количество числовых признаков:", len(num_cols))
print("Количество категориальных признаков:", len(cat_cols))
print("Количество категориальных признаков с низкой кардинальностью:", len(cat_low_cols))
print("Количество категориальных признаков с высокой кардинальностью:", len(cat_high_cols))

display(card.head(20))
display(card.tail(20))

In [ ]:
n_samples = X_train.shape[0]

for col in cat_high_cols:
    ratio = X_train[col].nunique() / n_samples
    print(col, "unique_ratio:", round(ratio, 4))

Был рассчитан показатель уникальности признаков как отношение числа уникальных значений к размеру обучающей выборки.

Признак `device_ip` имеет долю уникальных значений 0.84, что означает, что большинство значений являются практически уникальными для отдельных наблюдений.

Использование подобных признаков приводит к следующим проблемам:
- модель начинает запоминать отдельные объекты (риск переобучения),
- ухудшается обобщающая способность на новых данных,
- Target Encoding фактически превращается в запоминание.

В связи с этим признак `device_ip` был исключён из дальнейшего моделирования.

Остальные признаки высокой кардинальности имеют существенно меньшую долю уникальных значений и будут обработаны с помощью Target Encoding.

In [ ]:
X_train = X_train.drop(columns=["device_ip"])
X_cal   = X_cal.drop(columns=["device_ip"])
X_test  = X_test.drop(columns=["device_ip"])

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()

card = X_train[cat_cols].nunique()

thr = 20
cat_high_cols = card[card > thr].index.tolist()
cat_low_cols = card[card <= thr].index.tolist()

print("Количество числовых признаков:", len(num_cols))
print("Количество категориальных признаков с низкой кардинальностью:", len(cat_low_cols))
print("Количество категориальных признаков с высокой кардинальностью:", len(cat_high_cols))
print("Признаки с высокой кардинальностью:", cat_high_cols)

In [ ]:
def make_target_encoder():
    try:
        return TargetEncoder(handle_unknown="value", handle_missing="value")
    except Exception:
        import category_encoders as ce
        return ce.TargetEncoder(handle_unknown="value", handle_missing="value")

<div class="alert alert-info">

<h2>Комментарий студента<a class="tocSkip"></a></h2>

<b>Исправлено:</b><br/>
Все импорты перенесены.  

</div>

In [ ]:
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_low_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse=False)),
])

cat_high_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("target_enc", make_target_encoder()),
    ("post_imputer", SimpleImputer(strategy="median")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat_low", cat_low_pipe, cat_low_cols),
        ("cat_high", cat_high_pipe, cat_high_cols),
    ],
    remainder="drop"
)

print(preprocess)

In [ ]:
X_train_tr = preprocess.fit_transform(X_train, y_train)
X_test_tr = preprocess.transform(X_test)

print("Train shape:", X_train_tr.shape)
print("Test shape:", X_test_tr.shape)
print("NaN train:", np.isnan(X_train_tr).sum())
print("NaN test:", np.isnan(X_test_tr).sum())
print("Inf train:", np.isinf(X_train_tr).sum())
print("Inf test:", np.isinf(X_test_tr).sum())

Для предобработки данных построены отдельные пайплайны для числовых и категориальных признаков и объединены через `ColumnTransformer`.

**Числовые признаки:**
- пропуски заполняются медианой (`SimpleImputer(strategy="median")`), так как медиана устойчива к выбросам и асимметричным распределениям,
- признаки масштабируются (`StandardScaler`) для приведения к сопоставимому масштабу, что важно для линейных моделей.

**Категориальные признаки:**
- пропуски заполняются наиболее частым значением (`SimpleImputer(strategy="most_frequent")`),
- признаки с низкой кардинальностью кодируются через `OneHotEncoder`, так как число категорий невелико и размерность не растет критично,
- признаки с высокой кардинальностью кодируются через `Target Encoding`, чтобы избежать "взрыва" размерности при One-Hot Encoding.

После применения предобработки были обнаружены NaN в выходных признаках. Локализация показала, что они возникают только после `Target Encoding` (возможны редкие или невидимые на обучении категории). Для гарантии отсутствия пропусков перед обучением модели после `Target Encoding` добавлен дополнительный шаг иммутации (`SimpleImputer(strategy="median")`).

Поскольку предобработка реализована через `Pipeline` и `ColumnTransformer`, все статистики (медианы, частоты, параметры кодировщиков и масштаба) вычисляются только на обучающей выборке:
- для train используется `fit_transform`,
- для test используется только `transform`.

Это исключает утечки данных (data leakage). Результирующие матрицы имеют одинаковую размерность и не содержат NaN/Inf.

## 5. Отбор признаков

#### 5.1 Примените фильтрационные методы
- Посчитайте корреляцию каждого признака с целевой переменной.
- Отберите топ лучших признаков. Объясните, почему остановились именно на таком количестве признаков.
- Удалите признаки с очень низкой вариацией `VarianceThreshold`.

#### 5.2 Примените методы-обёртки
- Используйте методы-обёртки для поиска оптимального набора признаков.

#### 5.3 Выберите финальный набор признаков
- Объедините результаты методов.
- Выберите признаки, которые прошли фильтрацию.

In [ ]:
X_train_df = pd.DataFrame(X_train_tr)

corr = X_train_df.apply(lambda col: np.corrcoef(col, y_train)[0, 1])

corr_abs = corr.abs().sort_values(ascending=False)

print("Top 15 features by |correlation|:")
print(corr_abs.head(15))

print("\nBottom 15 features by |correlation|:")
print(corr_abs.tail(15))

In [ ]:
threshold = 0.02

selected_by_corr = corr_abs[corr_abs > threshold].index.tolist()

print("Selected by correlation:", len(selected_by_corr))
print("Total features:", X_train_df.shape[1])

Максимальная абсолютная корреляция признаков с целевой переменной составляет около 0.15, что типично для задач кликабельности (CTR), где сильные линейные зависимости встречаются редко.

Выбор количества признаков по фиксированному числу (например, топ-10) был бы произвольным, так как распределение корреляций плавное.

Поэтому был выбран порог |corr| > 0.02.
Признаки с корреляцией ниже этого значения считаются практически неинформативными в линейной модели и исключаются.

In [ ]:
X_train_corr = X_train_df[selected_by_corr]

vt = VarianceThreshold(threshold=1e-4)
X_train_vt = vt.fit_transform(X_train_corr)

mask = vt.get_support()
selected_by_vt = X_train_corr.columns[mask].tolist()

print("Количество признаков после VarianceThreshold:", len(selected_by_vt))
print("Удалено признаков из-за низкой вариации:", len(selected_by_corr) - len(selected_by_vt))

После фильтрации по корреляции дополнительно применён `VarianceThreshold` для удаления признаков с очень низкой дисперсией.
Такие признаки близки к константам и, как правило, не несут полезной информации для модели.

При пороге `threshold=1e-4` признаки с низкой вариацией не обнаружены: все 30 признаков, отобранных по корреляции, имеют достаточную дисперсию и сохраняются для следующего этапа.

In [ ]:
X_wrap = X_train_df[selected_by_vt]

estimator = LogisticRegression(
    penalty="l2",
    solver="liblinear",
    max_iter=1000,
    random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rfecv = RFECV(
    estimator=estimator,
    step=1,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

rfecv.fit(X_wrap, y_train)

selected_wrap = X_wrap.columns[rfecv.support_].tolist()

print("Оптимальное количество признаков:", rfecv.n_features_)

In [ ]:
final_features = selected_wrap

print("Финальное количество признаков:", len(final_features))

Применение метода-обёртки (L1-регуляризация)

Для поиска оптимального поднабора признаков применена логистическая регрессия с L1-регуляризацией.

L1-регуляризация выполняет встроенный отбор признаков, зануляя коэффициенты наименее информативных переменных.

При значении C=0.1 модель сократила количество признаков с 30 до 27. 
Дальнейшее усиление регуляризации (C=0.05) не привело к дополнительному сокращению, что говорит о стабильности оставшихся признаков.

Таким образом, финальный набор включает 27 признаков, прошедших как фильтрационный отбор, так и метод-обёртку.

<div class="alert alert-info">

<h2>Комментарий студента<a class="tocSkip"></a></h2>

<b>Исправлено:</b><br/>
Вместо L1-регуляризации реализован метод-обертка RFECV (Recursive Feature Elimination with Cross-Validation).  

Метод выполняет рекурсивное исключение признаков с оценкой качества модели через StratifiedKFold (5 фолдов) по метрике ROC-AUC.  

В результате оптимальный набор сформирован из 29 признаков.  
Финальный список признаков получен путем последовательного применения фильтрационных методов (корреляция + VarianceThreshold) и метода-обертки RFECV.

</div>

## 6. Обучение базовой модели

### 6.1 Обучите `DummyClassifier`
- Это нужно, чтобы обозначить самый простой базовый уровень работы модели.

### 6.2 Обучите `LogisticRegression`
- Используйте для обучения отобранные признаки.
- Примените кросс-валидацию на 5 фолдах.
- Посчитайте метрику PR-AUC. При необходимости дополнительно рассчитайте Precision, Recall и F1-score.
- Напоминаем, что для корректной кросс-валидации, предобработку нужно объединить с классификатором в Pipeline.

### 6.3 Обучите `SVC`

- Обучите SVC линейным ядром.
- Примените кросс-валидацию на 5 фолдах и посчитайте ту же метрику PR-ROC. При необходимости дополнительно рассчитайте Precision, Recall и F1-score.
- Калибровку модели мы проведём далее, поэтому здесь нужна модель `probability=False`

### 6.4 Сравните модели
- Убедитесь, что `LogisticRegression` работает лучше `DummyClassifier`.
- Сравните качество `LogisticRegression` с `SVC`.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

dummy_model = DummyClassifier(strategy="most_frequent", random_state=42)

scores_dummy = cross_validate(
    dummy_model,
    X_train,
    y_train,
    cv=cv,
    scoring={"pr_auc": "average_precision"},
    n_jobs=-1,
    return_train_score=False
)

print("Dummy PR-AUC (mean):", scores_dummy["test_pr_auc"].mean())
print("Dummy PR-AUC (std):", scores_dummy["test_pr_auc"].std())

<div class="alert alert-info">

<h2>Комментарий студента<a class="tocSkip"></a></h2>

<b>Исправлено:</b><br/>
Для DummyClassifier удален препроцессор из пайплайна, поскольку константная модель полностью игнорирует признаки X и не требует предобработки данных.

</div>

В качестве базового уровня обучен `DummyClassifier` со стратегией `most_frequent`.

Модель всегда предсказывает наиболее частый класс, что позволяет определить минимальный уровень качества.

Полученное значение PR-AUC ≈ 0.172 отражает базовую частоту положительного класса и служит ориентиром для сравнения более сложных моделей.

In [ ]:
class ColumnSelectorByIndex(BaseEstimator, TransformerMixin):
    def __init__(self, idx):
        self.idx = idx

    def fit(self, X, y=None):
        self.idx_ = np.array(self.idx, dtype=int)
        return self

    def transform(self, X):
        return X[:, self.idx_]

In [ ]:
selected_idx = list(map(int, selected_wrap))

lr_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("select", ColumnSelectorByIndex(selected_idx)),
    ("model", LogisticRegression(max_iter=500, solver="liblinear"))
])

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_lr = cross_validate(
    lr_pipe,
    X_train,
    y_train,
    cv=cv,
    scoring={"pr_auc": "average_precision"},
    n_jobs=-1,
    return_train_score=False
)

print("LogReg PR-AUC (mean):", scores_lr["test_pr_auc"].mean())
print("LogReg PR-AUC (std):", scores_lr["test_pr_auc"].std())

Логистическая регрессия обучалась на отобранных признаках (после этапа feature selection).
Для корректной кросс-валидации предобработка и модель объединены в `Pipeline`, что исключает утечки данных: все параметры предобработки обучаются только на тренировочной части каждого фолда.

Оценка проводилась на 5 фолдах (StratifiedKFold), целевая метрика - PR-AUC (average precision), так как задача имеет дисбаланс классов.

LogisticRegression значительно превосходит базовый уровень DummyClassifier по PR-AUC.

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

svc_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("select", ColumnSelectorByIndex(selected_idx)),
    ("model", LinearSVC(C=1.0, max_iter=5000, tol=1e-3))
])

scores_svc = cross_validate(
    svc_pipe,
    X_train,
    y_train,
    cv=cv5,
    scoring={"pr_auc": "average_precision"},
    n_jobs=-1,
    return_train_score=False
)

print("LinearSVC PR-AUC (mean):", scores_svc["test_pr_auc"].mean())
print("LinearSVC PR-AUC (std):", scores_svc["test_pr_auc"].std())

Сравнение моделей проводилось по метрике PR-AUC на 5 фолдах стратифицированной кросс-валидации.

Результаты:

- DummyClassifier: PR-AUC ≈ 0.172
- LogisticRegression: PR-AUC ≈ 0.364
- LinearSVC: PR-AUC ≈ 0.361

Логистическая регрессия значительно превосходит базовый уровень DummyClassifier, что подтверждает наличие обучаемого сигнала в данных.

По сравнению с LinearSVC логистическая регрессия показывает немного более высокое значение PR-AUC, при этом:
- быстрее обучается,
- не имеет проблем со сходимостью,
- предоставляет вероятности классов.

В качестве базовой модели для дальнейшего развития проекта выбрана LogisticRegression.

## 7. Подбор гиперпараметров: Grid Search с кросс-валидацией

#### 7.1 Определите сетку гиперпараметров
Определите ключевые параметры, которые влияют на качество моделей `LogisticRegression` и `SVC`.

#### 7.2 Примените Grid Search
- Используйте `GridSearchCV` для перебора всех комбинаций.
- Используйте `scoring='average_precision'`.
- Выведите лучшие параметры и их метрики.

#### 7.3 Составьте таблицу результатов
- Покажите топ-10 конфигураций с их метриками.

In [ ]:
param_grid_lr = {
    "model__C": [0.01, 0.05, 0.1, 0.5, 1, 5],
    "model__penalty": ["l1", "l2"],
    "model__class_weight": [None, "balanced"]
}

param_grid_svc = {
    "model__C": [0.05, 0.1, 0.5, 1.0],
    "model__class_weight": [None, "balanced"]
}

In [ ]:
lr_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("select", ColumnSelectorByIndex(selected_idx)),
    ("model", LogisticRegression(
        solver="liblinear",
        max_iter=500
    ))
])

grid_lr = GridSearchCV(
    lr_pipe,
    param_grid=param_grid_lr,
    scoring="average_precision",
    cv=cv5,
    n_jobs=-1,
    verbose=1
)

grid_lr.fit(X_train, y_train)

print("Лучшие параметры (LogReg):", grid_lr.best_params_)
print("Лучший PR-AUC:", grid_lr.best_score_)

In [ ]:
svc_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("select", ColumnSelectorByIndex(selected_idx)),
    ("model", LinearSVC(max_iter=5000, tol=1e-3))
])

grid_svc = GridSearchCV(
    svc_pipe,
    param_grid=param_grid_svc,
    scoring="average_precision",
    cv=cv5,
    n_jobs=-1,
    verbose=1
)

grid_svc.fit(X_train, y_train)

print("Лучшие параметры (LinearSVC):", grid_svc.best_params_)
print("Лучший PR-AUC:", grid_svc.best_score_)

In [ ]:
def top10_from_grid(grid, model_name):
    df = pd.DataFrame(grid.cv_results_).copy()
    df = df.sort_values("mean_test_score", ascending=False)
    out = df[["rank_test_score", "mean_test_score", "std_test_score", "params"]].head(10)
    out.insert(0, "model", model_name)
    out = out.rename(columns={
        "mean_test_score": "mean_pr_auc",
        "std_test_score": "std_pr_auc"
    })
    return out

In [ ]:
top_lr = top10_from_grid(grid_lr, "LogisticRegression")
top_svc = top10_from_grid(grid_svc, "LinearSVC")

top_all = pd.concat([top_lr, top_svc], ignore_index=True)
top_all = top_all.sort_values("mean_pr_auc", ascending=False).head(10)

top_all

По результатам GridSearchCV на 5-fold кросс-валидации лучшей моделью стала LogisticRegression с параметрами:

- C = 0.1
- penalty = l2
- class_weight = None

Лучшее значение метрики PR-AUC ≈ 0.364.

LinearSVC показал немного более низкое качество (≈ 0.362), несмотря на подбор гиперпараметров.

Таким образом, финальной моделью для дальнейшего анализа и использования выбрана LogisticRegression.

## 8. Финальная модель

#### 8.1 Обучите финальную модель
- Используйте лучшие параметры из Grid Search.
- Обучите модели на всей обучающей выборке.

#### 8.2 Посчитайте метрики на тестовой выборке
- Необходимые метрики:
  - PR-AUC.
  - Оценка Бриера.
  - Дополнительные метрики при необходимости.

#### 8.3 Проанализируйте веса модели
- Выведите самые важные признаки по модулю коэффициентов.
- Интерпретируйте результаты.

In [ ]:
final_pipe = grid_lr.best_estimator_

<div class="alert alert-info">

<h2>Комментарий студента<a class="tocSkip"></a></h2>

<b>Исправлено:</b><br/>
Удалено повторное обучение модели.  
В качестве финальной модели используется grid_lr.best_estimator_, 
который уже обучен на всех обучающих данных с оптимальными гиперпараметрами.

</div>

In [ ]:
y_proba_test = final_pipe.predict_proba(X_test)[:, 1]

pr_auc_test = average_precision_score(y_test, y_proba_test)

brier = brier_score_loss(y_test, y_proba_test)

y_pred_test = (y_proba_test >= 0.5).astype(int)

precision = precision_score(y_test, y_pred_test)
recall = recall_score(y_test, y_pred_test)
f1 = f1_score(y_test, y_pred_test)

print("PR-AUC (test):", pr_auc_test)
print("Brier score:", brier)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

In [ ]:
X_train_transformed = final_pipe.named_steps["preprocess"].transform(X_train)
X_train_selected = final_pipe.named_steps["select"].transform(X_train_transformed)

model = final_pipe.named_steps["model"]

coef = pd.Series(
    model.coef_[0],
    index=selected_idx
)

coef_abs = coef.abs().sort_values(ascending=False)

print("Топ-15 признаков по |весу|:")
print(coef_abs.head(15))

Финальная модель LogisticRegression с параметрами, полученными в ходе GridSearchCV (C=0.1, penalty=l2), обучена на всей обучающей выборке.

Оценка проведена на отложенной тестовой выборке.

- PR-AUC ≈ 0.362
- Brier score ≈ 0.129
- Precision ≈ 0.584
- Recall ≈ 0.064
- F1-score ≈ 0.116

Значение PR-AUC на тестовой выборке близко к результату кросс-валидации, что свидетельствует об отсутствии переобучения.

Низкое значение Recall связано с использованием стандартного порога 0.5, который не является оптимальным для задач с дисбалансом классов.

Анализ коэффициентов логистической регрессии показал, что наибольший вклад в предсказание оказывают признаки с максимальным абсолютным значением коэффициента.

Знак коэффициента отражает направление влияния признака на вероятность положительного класса, а модуль - силу влияния.

## 9. Калибровка модели

#### 9.1 Проверьте текущую калибровку
- Постройте калибровочную кривую, используйте `sklearn.calibration.calibration_curve`.
- Для обработки «сырых» значений SVC, нужно применить стандартную (необученную) сигмоиду для получения [0, 1].

#### 9.2 Примените методы калибровки
- Используйте `CalibratedClassifierCV` с методом `'isotonic'`.
- **Важно:** используйте для процедуры отдельную калибровочную выборку!

#### 9.3 Сравните модели до и после калибровки
- Посчитайте оценки Бриера для моделей до и после калибровки.
- Дополнительно можете рассчитать ECE и MCE для моделей до и после калибровки.
- Визуализируйте калибровочные кривые для моделей до и после калибровки.

<div class="alert alert-info">

<h2>Комментарий студента<a class="tocSkip"></a></h2>

<b>Исправлено:</b><br/>
Калибровочная выборка выделена на этапе разделения данных (вместе с тестовой) и является независимой от обучающей выборки.<br/>
Удалено повторное обучение модели перед калибровкой.<br/>
В качестве финальной используется grid_lr.best_estimator_, уже обученный на обучающей выборке с оптимальными гиперпараметрами.<br/>
Калибровка выполняется через CalibratedClassifierCV с параметром cv="prefit", что исключает утечку данных.

</div>

In [ ]:
print("X_train:", X_train.shape)
print("X_cal  :", X_cal.shape)

train_cols = set(X_train.columns)
cal_cols = set(X_cal.columns)

print("В train, но нет в cal:", sorted(train_cols - cal_cols))
print("В cal, но нет в train:", sorted(cal_cols - train_cols))

In [ ]:
svc_pipe = grid_svc.best_estimator_

In [ ]:
proba_lr_raw_cal = final_pipe.predict_proba(X_cal)[:, 1]

scores_svc_cal = svc_pipe.decision_function(X_cal)
proba_svc_raw_cal = expit(scores_svc_cal)

print("LR raw proba range:", float(proba_lr_raw_cal.min()), float(proba_lr_raw_cal.max()))
print("SVC raw(sigmoid) proba range:", float(proba_svc_raw_cal.min()), float(proba_svc_raw_cal.max()))

In [ ]:
def plot_calibration(y_true, curves, n_bins=10, strategy="quantile", title="Calibration"):
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], label="Perfectly calibrated")

    for name, proba in curves.items():
        frac_pos, mean_pred = calibration_curve(
            y_true, proba, n_bins=n_bins, strategy=strategy
        )
        plt.plot(mean_pred, frac_pos, marker="o", label=name)

    plt.xlabel("Mean predicted probability")
    plt.ylabel("Fraction of positives")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
plot_calibration(
    y_cal,
    {"LogReg raw": proba_lr_raw_cal, "LinearSVC raw(sigmoid)": proba_svc_raw_cal},
    n_bins=10,
    strategy="quantile",
    title="Calibration curves BEFORE calibration (on X_cal)"
)

Калибровочная кривая показывает, что:

- LogisticRegression демонстрирует достаточно хорошую калибровку. Кривая близка к диагонали.
- LinearSVC с применением стандартной сигмоиды имеет выраженное смещение и сжатый диапазон вероятностей.

Следовательно, LogisticRegression уже хорошо откалибрована, а LinearSVC требует дополнительной калибровки.

In [ ]:
lr_iso = CalibratedClassifierCV(final_pipe, method="isotonic", cv="prefit")
lr_iso.fit(X_cal, y_cal)

svc_iso = CalibratedClassifierCV(svc_pipe, method="isotonic", cv="prefit")
svc_iso.fit(X_cal, y_cal)

In [ ]:
proba_lr_cal = final_pipe.predict_proba(X_cal)[:, 1]
proba_lr_test = final_pipe.predict_proba(X_test)[:, 1]

scores_svc_cal = svc_pipe.decision_function(X_cal)
scores_svc_test = svc_pipe.decision_function(X_test)

proba_svc_sig_cal = expit(scores_svc_cal)
proba_svc_sig_test = expit(scores_svc_test)

print("LR cal range:", float(proba_lr_cal.min()), float(proba_lr_cal.max()))
print("SVC sigmoid cal range:", float(proba_svc_sig_cal.min()), float(proba_svc_sig_cal.max()))

In [ ]:
iso_lr = IsotonicRegression(out_of_bounds="clip")
iso_lr.fit(proba_lr_cal, y_cal)

iso_svc = IsotonicRegression(out_of_bounds="clip")
iso_svc.fit(scores_svc_cal, y_cal)

Для улучшения вероятностной интерпретации моделей была применена изотоническая регрессия.

Калибровка выполнялась:
- на отдельной калибровочной выборке (X_cal),
- без использования тестовой выборки,
- отдельно для каждой модели.

Для LogisticRegression калибровка применялась к предсказанным вероятностям.
Для LinearSVC калибровка применялась к значениям decision_function.

In [ ]:
proba_lr_iso_test = iso_lr.transform(proba_lr_test)
proba_svc_iso_test = iso_svc.transform(scores_svc_test)

def row(name, y_true, proba):
    return {
        "model": name,
        "PR-AUC": average_precision_score(y_true, proba),
        "Brier": brier_score_loss(y_true, proba)
    }

metrics_df = pd.DataFrame([
    row("LogReg raw", y_test, proba_lr_test),
    row("LogReg isotonic", y_test, proba_lr_iso_test),
    row("LinearSVC raw(sigmoid)", y_test, proba_svc_sig_test),
    row("LinearSVC isotonic(score)", y_test, proba_svc_iso_test),
])

display(metrics_df)

Результаты на тестовой выборке показали:

- LogisticRegression практически не изменила Brier score, что говорит о хорошей исходной калибровке.
- Для LinearSVC Brier score значительно улучшился после калибровки.
- PR-AUC немного снизился, что объясняется возможным нарушением ранжирования при изотонической калибровке.

Вывод:
LogisticRegression остается финальной моделью благодаря стабильному качеству и хорошей вероятностной интерпретации.

In [ ]:
def plot_calibration_curves(y_true, curves, title, n_bins=10, strategy="quantile"):
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], label="Perfectly calibrated")

    for name, proba in curves.items():
        frac_pos, mean_pred = calibration_curve(
            y_true, proba, n_bins=n_bins, strategy=strategy
        )
        plt.plot(mean_pred, frac_pos, marker="o", label=name)

    plt.xlabel("Mean predicted probability")
    plt.ylabel("Fraction of positives")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

plot_calibration_curves(
    y_test,
    {"LogReg raw": proba_lr_test, "LogReg isotonic": proba_lr_iso_test},
    title="Calibration curves: LogisticRegression (before vs after)"
)

plot_calibration_curves(
    y_test,
    {"LinearSVC raw(sigmoid)": proba_svc_sig_test, "LinearSVC isotonic(score)": proba_svc_iso_test},
    title="Calibration curves: LinearSVC (before vs after)"
)

Графики показывают:

- Для LogisticRegression изотоническая калибровка практически не изменила форму кривой. Модель уже демонстрировала хорошую согласованность вероятностей с реальной долей положительного класса.

- Для LinearSVC калибровка существенно улучшила форму кривой. После применения isotonic регрессии предсказанные вероятности стали значительно ближе к идеально откалиброванной диагонали.

Таким образом, LogisticRegression изначально обладала хорошей вероятностной интерпретацией, тогда как LinearSVC потребовала дополнительной калибровки.

## 10. Оценка качества калибровки

#### 10.1 Посчитайте метрики калибровки
- Оценка Бриера — средняя ошибка предсказанной вероятности.
- Дополнительная метрика ECE: среднее расхождение вероятностей.
- Дополнительная метрика MCE: максимальное расхождение вероятностей.

#### 10.2 Сравните модели до и после калибровки
- Выведите все метрики в одной таблице.
- Сделайте вывод о том, улучшила ли калибровка качество моделей.

In [ ]:
def ece_mce(y_true, proba, n_bins=10):
    y_true = np.asarray(y_true)
    proba = np.asarray(proba)

    bins = np.linspace(0, 1, n_bins + 1)
    bin_ids = np.digitize(proba, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    ece = 0.0
    mce = 0.0
    n = len(y_true)

    for b in range(n_bins):
        mask = (bin_ids == b)
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = proba[mask].mean()
        gap = abs(acc - conf)

        ece += (mask.sum() / n) * gap
        mce = max(mce, gap)

    return ece, mce

In [ ]:
rows = []
for name, proba in [
    ("LogReg raw", proba_lr_test),
    ("LogReg isotonic", proba_lr_iso_test),
    ("LinearSVC raw(sigmoid)", proba_svc_sig_test),
    ("LinearSVC isotonic(score)", proba_svc_iso_test),
]:
    ece, mce = ece_mce(y_test, proba, n_bins=10)
    rows.append({
        "model": name,
        "Brier": brier_score_loss(y_test, proba),
        "ECE": ece,
        "MCE": mce
    })

calibration_table = pd.DataFrame(rows).sort_values("Brier")
display(calibration_table)

Рассчитаны следующие метрики:

- **Brier score** - средняя квадратичная ошибка вероятностей.
- **ECE (Expected Calibration Error)** - среднее расхождение между предсказанной вероятностью и фактической долей положительного класса.
- **MCE (Maximum Calibration Error)** - максимальное расхождение по бинам.

Результаты показывают:

- **LogisticRegression (raw)** демонстрирует наилучшие показатели калибровки:
  - минимальный Brier,
  - минимальный ECE,
  - низкий MCE.

- Применение isotonic к LogisticRegression не улучшило метрики, а слегка их ухудшило. Это подтверждает, что модель изначально была хорошо откалибрована.

- Для LinearSVC наблюдается существенное улучшение после калибровки:
  - Brier снизился с 0.162 до ~0.130,
  - ECE снизился с 0.170 до ~0.008,
  - MCE значительно уменьшился.

Таким образом, изотоническая калибровка значительно улучшила вероятностную интерпретацию LinearSVC, но практически не повлияла на LogisticRegression.

## 11. Финальный отчёт и выводы

### 11.1 Сведите все результаты в таблицу

Покажите:
- Характеристики базовой модели `DummyClassifier`.
- Характеристики финальной модели.
- Метрики до и после калибровки.
- Топ-5 самых важных признаков.

### 11.2 Напишите выводы

Ответьте на вопросы:
- Улучшилось ли качество модели по сравнению с базовой?
- Какие признаки больше всего влияют на вероятность клика?
- Насколько хорошо модель откалибрована?
- Готова ли модель к использованию в продакшене?

### 11.3 Рекомендации

- Какие возможности улучшения модели вы видите?

In [ ]:
dummy = DummyClassifier(strategy="prior")
dummy.fit(X_train, y_train)

proba_dummy_test = dummy.predict_proba(X_test)[:, 1]

dummy_pr_auc = average_precision_score(y_test, proba_dummy_test)
dummy_brier = brier_score_loss(y_test, proba_dummy_test)

print("Dummy PR-AUC:", dummy_pr_auc)
print("Dummy Brier:", dummy_brier)

In [ ]:
final_summary = pd.DataFrame([
    {
        "model": "DummyClassifier",
        "PR-AUC": dummy_pr_auc,
        "Brier": dummy_brier,
        "ECE": None
    },
    {
        "model": "LogReg raw",
        "PR-AUC": average_precision_score(y_test, proba_lr_test),
        "Brier": brier_score_loss(y_test, proba_lr_test),
        "ECE": ece_mce(y_test, proba_lr_test)[0]
    },
    {
        "model": "LogReg isotonic",
        "PR-AUC": average_precision_score(y_test, proba_lr_iso_test),
        "Brier": brier_score_loss(y_test, proba_lr_iso_test),
        "ECE": ece_mce(y_test, proba_lr_iso_test)[0]
    },
    {
        "model": "LinearSVC raw(sigmoid)",
        "PR-AUC": average_precision_score(y_test, proba_svc_sig_test),
        "Brier": brier_score_loss(y_test, proba_svc_sig_test),
        "ECE": ece_mce(y_test, proba_svc_sig_test)[0]
    },
    {
        "model": "LinearSVC isotonic",
        "PR-AUC": average_precision_score(y_test, proba_svc_iso_test),
        "Brier": brier_score_loss(y_test, proba_svc_iso_test),
        "ECE": ece_mce(y_test, proba_svc_iso_test)[0]
    },
]).sort_values("PR-AUC", ascending=False)

display(final_summary)

В таблице представлены:

- Базовая модель DummyClassifier
- Финальная LogisticRegression (до и после калибровки)
- LinearSVC (до и после калибровки)
- Метрики PR-AUC, Brier и ECE

Основные наблюдения:

- Финальная LogisticRegression значительно превосходит DummyClassifier по PR-AUC (0.355 против 0.172).
- LogisticRegression имеет наименьший Brier score и минимальный ECE, что говорит о хорошем качестве калибровки.
- LinearSVC без калибровки демонстрирует высокую ошибку калибровки (ECE ≈ 0.17).
- После изотонической калибровки LinearSVC существенно улучшает Brier и ECE, но всё равно уступает LogisticRegression по PR-AUC.

**1. Улучшилось ли качество по сравнению с базовой моделью?**

Да. Финальная LogisticRegression более чем в два раза превосходит DummyClassifier по PR-AUC (0.355 против 0.172), что подтверждает способность модели выявлять закономерности в данных.

**2. Какие признаки больше всего влияют на вероятность клика?**

Наиболее значимыми признаками являются поведенческие и категориальные признаки, связанные с:
- характеристиками устройства,
- параметрами размещения баннера,
- агрегированными ML-признаками.

Именно они формируют основной вклад в вероятность клика.

**3. Насколько хорошо модель откалибрована?**

LogisticRegression демонстрирует хорошую калибровку:
- минимальный Brier score,
- минимальный ECE,
- калибровочная кривая близка к диагонали.

Дополнительная калибровка не привела к улучшению качества, что подтверждает корректность исходных вероятностей.

**4. Готова ли модель к использованию в продакшене?**

Да, модель можно считать готовой к использованию:
- она существенно превосходит базовую,
- демонстрирует стабильные вероятности,
- обладает интерпретируемыми коэффициентами,
- устойчива к переобучению.

Для дальнейшего повышения качества модели можно рассмотреть:

- Добавление более сложных моделей (Gradient Boosting, CatBoost).
- Расширение набора признаков (временные признаки, взаимодействия).
- Более тонкую настройку регуляризации.
- Использование кросс-валидационной калибровки.
- Оптимизацию порога классификации под бизнес-метрику.

Также возможно внедрение онлайн-мониторинга качества и калибровки модели в продакшене.

## 12. Сохранение модели для продакшена

### 12.1 Сохраните артефакты

Сохраните:
1. пайплайн предобработки данных `preprocessor`;
2. финальную модель `calibrated_model`;
3. информацию о выбранных признаках.

### 12.2 Проверьте работоспособность вашего кода

- Загрузите сохранённые артефакты.
- Сделайте предсказания на новых данных.
- Убедитесь, что результаты совпадают.

In [ ]:
preprocessor = final_pipe.named_steps["preprocess"]

calibrated_model = final_pipe

selected_features = final_pipe.named_steps["select"].idx

joblib.dump(preprocessor, "preprocessor.joblib")
joblib.dump(calibrated_model, "calibrated_model.joblib")

with open("selected_features.json", "w") as f:
    json.dump(list(selected_features), f)

In [ ]:
preprocessor_loaded = joblib.load("preprocessor.joblib")
model_loaded = joblib.load("calibrated_model.joblib")

with open("selected_features.json", "r") as f:
    selected_features_loaded = json.load(f)

print("Loaded selected_features length:", len(selected_features_loaded))

X_new = X_test.iloc[:5].copy()

proba_orig = calibrated_model.predict_proba(X_new)[:, 1]

proba_loaded = model_loaded.predict_proba(X_new)[:, 1]

print("orig:", np.round(proba_orig, 6))
print("load:", np.round(proba_loaded, 6))
print("allclose:", np.allclose(proba_orig, proba_loaded))